# Data Outlier Audit

This notebook audits seeded benchmark parameter datasets for distribution drift, missing values, duplicates, and numeric outliers before full training.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
ROOT = Path("..").resolve()
DATASETS = {
    "train": ROOT / "data/static/scenario_parameter_records_seeded_train.json",
    "val": ROOT / "data/static/scenario_parameter_records_seeded_val.json",
    "holdout": ROOT / "data/static/scenario_parameter_records_seeded_holdout.json",
}
NUMERIC_COLUMNS = [
    "base_spread_prob",
    "wind_strength",
    "spread_rate_1h_m",
    "spread_score",
    "weather_score",
    "cffdrs_dryness_score",
    "size_factor",
    "fire_type_factor",
    "fuel_factor",
    "rain_factor",
]

In [ ]:
def load_records(path: Path, split_name: str) -> pd.DataFrame:
    payload = json.loads(path.read_text())
    records = payload.get("records", []) if isinstance(payload, dict) else payload
    frame = pd.DataFrame(records)
    frame["split_source"] = split_name
    return frame

frames = [load_records(path, split) for split, path in DATASETS.items()]
df = pd.concat(frames, ignore_index=True)
df.head()

In [ ]:
summary = {
    "rows_total": int(len(df)),
    "rows_by_split_source": df["split_source"].value_counts(dropna=False).to_dict(),
    "rows_by_split_field": df["split"].value_counts(dropna=False).to_dict(),
    "unique_record_id": int(df["record_id"].nunique(dropna=True)),
    "duplicate_record_id_count": int(df.duplicated(subset=["record_id"]).sum()),
    "duplicate_fire_id_count": int(df.duplicated(subset=["fire_id"]).sum()),
}
summary

In [ ]:
required = [
    "record_id",
    "split",
    "base_spread_prob",
    "severity_bucket",
    "wind_direction",
    "wind_strength",
    "ignition_seed",
    "layout_seed",
]
null_counts = df[required].isna().sum().sort_values(ascending=False)
null_counts

In [ ]:
for col in NUMERIC_COLUMNS:
    if col in df.columns:
        fig, ax = plt.subplots(1, 2, figsize=(13, 4))
        sns.histplot(data=df, x=col, hue="split_source", kde=True, ax=ax[0], stat="density", common_norm=False)
        sns.boxplot(data=df, x="split_source", y=col, ax=ax[1])
        ax[0].set_title(f"Distribution: {col}")
        ax[1].set_title(f"Boxplot by split: {col}")
        plt.tight_layout()
        plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.countplot(data=df, x="severity_bucket", hue="split_source", ax=axes[0])
axes[0].set_title("Severity bucket by split")
sns.countplot(data=df, x="wind_direction", hue="split_source", ax=axes[1])
axes[1].set_title("Wind direction by split")
sns.countplot(data=df, x="record_quality_flag", hue="split_source", ax=axes[2])
axes[2].set_title("Record quality flag by split")
for ax in axes:
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
def iqr_outlier_mask(series: pd.Series) -> pd.Series:
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    if iqr == 0 or np.isnan(iqr):
        return pd.Series(False, index=series.index)
    lo = q1 - 1.5 * iqr
    hi = q3 + 1.5 * iqr
    return (series < lo) | (series > hi)

outlier_flags = pd.DataFrame(index=df.index)
for col in NUMERIC_COLUMNS:
    if col in df.columns:
        outlier_flags[col] = iqr_outlier_mask(pd.to_numeric(df[col], errors="coerce"))

df_outliers = df[outlier_flags.any(axis=1)].copy()
df_outliers["outlier_feature_count"] = outlier_flags.sum(axis=1)
outlier_summary = outlier_flags.sum().sort_values(ascending=False)
outlier_summary

In [ ]:
cols = ["record_id", "fire_id", "split", "split_source", "severity_bucket", "wind_direction", "outlier_feature_count"]
numeric_present = [c for c in NUMERIC_COLUMNS if c in df_outliers.columns]
display(df_outliers[cols + numeric_present].sort_values("outlier_feature_count", ascending=False).head(50))

In [ ]:
output_dir = ROOT / "outputs/data_audit"
output_dir.mkdir(parents=True, exist_ok=True)

(output_dir / "summary_stats.json").write_text(json.dumps(summary, indent=2))
df.describe(include="all", datetime_is_numeric=True).to_csv(output_dir / "describe_all.csv")
df_outliers.to_csv(output_dir / "outlier_rows.csv", index=False)
outlier_summary.to_csv(output_dir / "outlier_feature_counts.csv")
print(f"Wrote data audit artifacts to {output_dir}")

## Go/No-Go Checklist

- No missing required fields in seeded split datasets
- No unexpected split leakage between `split_source` and `split`
- Numeric ranges are physically plausible (`base_spread_prob` in [0,1], `wind_strength` in [0,1])
- Outliers are explainable and not caused by parsing/ETL bugs
- Severity and wind distributions are plausible across train/val/holdout